In [ ]:
# Import configuration and helper functions
%run ../config.py
%run ../resources/data_setup_functions

import sys
import time
import logging
import os
from pathlib import Path

from databricks.sdk import WorkspaceClient
from databricks.sdk.runtime import dbutils

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

# Step 1: Create catalog & schema
create_catalog_and_schema(spark, catalog, schema)

# Step 2: Create volume for raw documents
create_volume(spark, catalog, schema, volume_name)

# Step 3: Create table for document metadata
create_table(spark, catalog, schema, table)

# Step 4: Upload first file from data/10k folder
data_folder = "/data"
local_file_path = os.getcwd()+data_folder
local_root = local_file_path  # path to "data/"

upload_file_to_volume(spark, local_file_path, catalog, schema, volume_name, table)

# Step 5: Create tables for volume subfolders (inline implementation)
logger.info("Creating tables for volume subfolders...")

volume_path = f"/Volumes/{catalog}/{schema}/{volume_name}"
subfolders = dbutils.fs.ls(volume_path)

create_tables_from_volume_subfolders(spark, catalog, schema, volume_name, logger)